<a href="https://colab.research.google.com/github/sijianing4/DSAI5201/blob/main/train%EF%BC%88llama%2Bqwen%EF%BC%89.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers accelerate sentencepiece pandas -q

In [ ]:
#Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!ls "/content/drive/MyDrive/DSAI5201"

baseline_checkpoint.csv		    full_cautious_outputs（qwen）.csv
cautious_checkpoint.csv		    sampled_reasoning_fake.csv
cautious_checkpoint（llama）.csv    sampled_reasoning_FCT.csv
full_baseline_outputs（llama）.csv  sampled_reasoning_nota.csv
full_baseline_outputs（qwen）.csv   train（llama+qwen）.ipynb
full_cautious_outputs（llama）.csv  uet_full_50_for_experiment.csv


In [ ]:
# Set data path
import os
BASE_PATH = "/content/drive/MyDrive/DSAI5201"
FCT_FILE = os.path.join(BASE_PATH, "sampled_reasoning_FCT.csv")
FAKE_FILE = os.path.join(BASE_PATH, "sampled_reasoning_fake.csv")
NOTA_FILE = os.path.join(BASE_PATH, "sampled_reasoning_nota.csv")
UET_FILE = os.path.join(BASE_PATH, "uet_full_50_for_experiment.csv")
print(FCT_FILE)
print(FAKE_FILE)
print(NOTA_FILE)
print(UET_FILE)

/content/drive/MyDrive/DSAI5201/sampled_reasoning_FCT.csv
/content/drive/MyDrive/DSAI5201/sampled_reasoning_fake.csv
/content/drive/MyDrive/DSAI5201/sampled_reasoning_nota.csv
/content/drive/MyDrive/DSAI5201/uet_full_50_for_experiment.csv


In [ ]:
# Check if the file exists
for path in [FCT_FILE, FAKE_FILE, NOTA_FILE, UET_FILE]:
    print(path, "->", os.path.exists(path))

/content/drive/MyDrive/DSAI5201/sampled_reasoning_FCT.csv -> True
/content/drive/MyDrive/DSAI5201/sampled_reasoning_fake.csv -> True
/content/drive/MyDrive/DSAI5201/sampled_reasoning_nota.csv -> True
/content/drive/MyDrive/DSAI5201/uet_full_50_for_experiment.csv -> True


In [ ]:
# read data
import pandas as pd
fct_df = pd.read_csv(FCT_FILE)
fake_df = pd.read_csv(FAKE_FILE)
nota_df = pd.read_csv(NOTA_FILE)
uet_df = pd.read_csv(UET_FILE)

print("FCT:", fct_df.shape)
print("fake:", fake_df.shape)
print("nota:", nota_df.shape)
print("UET:", uet_df.shape)

FCT: (200, 13)
fake: (200, 13)
nota: (200, 13)
UET: (50, 13)


In [ ]:
# View column names
print("FCT columns:", fct_df.columns.tolist())
print("fake columns:", fake_df.columns.tolist())
print("nota columns:", nota_df.columns.tolist())
print("UET columns:", uet_df.columns.tolist())

FCT columns: ['id', 'dataset', 'question', 'options', 'subject_name', 'topic_name', 'correct_answer', 'correct_index', 'expected_behavior', 'student_answer', 'student_index', 'missing_evidence', 'source_split']
fake columns: ['id', 'dataset', 'question', 'options', 'subject_name', 'topic_name', 'correct_answer', 'correct_index', 'expected_behavior', 'student_answer', 'student_index', 'missing_evidence', 'source_split']
nota columns: ['id', 'dataset', 'question', 'options', 'subject_name', 'topic_name', 'correct_answer', 'correct_index', 'expected_behavior', 'student_answer', 'student_index', 'missing_evidence', 'source_split']
UET columns: ['id', 'dataset', 'question', 'options', 'subject_name', 'topic_name', 'correct_answer', 'correct_index', 'expected_behavior', 'student_answer', 'student_index', 'missing_evidence', 'source_split']


In [ ]:
# Unified fields
REQUIRED_COLUMNS = [
    "id", "dataset", "question", "options", "subject_name", "topic_name",
    "correct_answer", "correct_index", "expected_behavior",
    "student_answer", "student_index", "missing_evidence", "source_split"
]
def ensure_columns(df, dataset_name):
    # Fill in the missing fields to avoid subsequent KeyErrors.
    for col in REQUIRED_COLUMNS:
        if col not in df.columns:
            df[col] = ""
    # Unified dataset fields
    if "dataset" in df.columns:
        df["dataset"] = df["dataset"].fillna("").replace("", dataset_name)
    else:
        df["dataset"] = dataset_name
    return df[REQUIRED_COLUMNS].copy()
fct_df = ensure_columns(fct_df, "fct")
fake_df = ensure_columns(fake_df, "fake")
nota_df = ensure_columns(nota_df, "nota")
uet_df = ensure_columns(uet_df, "uet")

In [ ]:
# Check again
print(fct_df.head(1).T)

                                                                   0
id                              d9b1db11-0952-48ee-8505-ec4af9c40aa9
dataset                                                          fct
question           We perform the determination of prostate-speci...
options            {'0': 'PSA sensitivity in adolescents will be ...
subject_name                                                medicine
topic_name                                                       NaN
correct_answer     The internal validity of PSA in healthy adoles...
correct_index                                                      2
expected_behavior                                             answer
student_answer     PSA sensitivity in adolescents will be higher ...
student_index                                                      0
missing_evidence                                                 NaN
source_split                                                     val


In [ ]:
# parse options
import ast
import json
import math
def parse_options(options_str):
    """
    Explanation in Chinese:
Parses the string "options" into a dictionary,
keeping only numeric keys (0, 1, 2, 3...) to avoid misinterpreting fields as options.
    """
    if pd.isna(options_str):
        return {}

    if isinstance(options_str, dict):
        raw = options_str
    else:
        try:
            raw = ast.literal_eval(options_str)
        except Exception:
            try:
                raw = json.loads(options_str)
            except Exception:
                return {}

    clean = {}
    for k, v in raw.items():
        if str(k).isdigit():
            clean[str(k)] = v
    return clean

In [ ]:
#  Extracting Misleading Answers from FCT
# Prioritize using the student_answer column;
# If empty, then try extracting from a special key in options.
def extract_student_answer(row):
    # First use a single field
    if "student_answer" in row and pd.notna(row["student_answer"]):
        val = str(row["student_answer"]).strip()
        if val:
            return val
    # If the misleading answer is hidden in a special key in the options, then retrieve it.
    try:
        raw = ast.literal_eval(row["options"]) if isinstance(row["options"], str) else row["options"]
        if isinstance(raw, dict):
            for key in ["correct answer", "misleading_answer", "suggested_answer"]:
                if key in raw:
                    return raw[key]
    except Exception:
        pass
    return ""

In [ ]:
# Format the options as A/B/C/D...
def format_options_for_prompt(options_dict):
    labels = ["A", "B", "C", "D", "E", "F"]
    lines = []
    sorted_keys = sorted(options_dict.keys(), key=lambda x: int(x))
    for i, key in enumerate(sorted_keys):
        if i < len(labels):
            lines.append(f"{labels[i]}. {options_dict[key]}")
    return "\n".join(lines)

In [ ]:
# Test the parsing effect
sample_row = fct_df.iloc[0]
sample_options = parse_options(sample_row["options"])
sample_student_answer = extract_student_answer(sample_row)
print("Parsed options:", sample_options)
print("Student answer:", sample_student_answer)
print("\nFormatted options:\n")
print(format_options_for_prompt(sample_options))

Parsed options: {'0': 'PSA sensitivity in adolescents will be higher than in the elderly.', '1': 'The specificity of PSA in adolescents will be lower than in the elderly.', '2': 'The internal validity of PSA in healthy adolescents or elderly will not change.', '3': 'The positive predictive value of PSA in adolescents will be higher than in the elderly.'}
Student answer: PSA sensitivity in adolescents will be higher than in the elderly.

Formatted options:

A. PSA sensitivity in adolescents will be higher than in the elderly.
B. The specificity of PSA in adolescents will be lower than in the elderly.
C. The internal validity of PSA in healthy adolescents or elderly will not change.
D. The positive predictive value of PSA in adolescents will be higher than in the elderly.


In [ ]:
# Run mode settings
# pilot: Run only the first few questions of each category
# full: Run the full dataset
RUN_MODE = "full"   # It can be changed to "pilot"

PILOT_SIZE = {
    "fct": 5,
    "fake": 5,
    "nota": 5,
    "uet": 10
}

def select_run_subset(df, dataset_name):
    # The number of questions to run depends on the pattern.
    if RUN_MODE == "pilot":
        return df.head(PILOT_SIZE[dataset_name]).copy()
    return df.copy()
run_fct_df = select_run_subset(fct_df, "fct")
run_fake_df = select_run_subset(fake_df, "fake")
run_nota_df = select_run_subset(nota_df, "nota")
run_uet_df = select_run_subset(uet_df, "uet")

run_df = pd.concat([run_fct_df, run_fake_df, run_nota_df, run_uet_df], ignore_index=True)

print("RUN MODE:", RUN_MODE)
print("Total run size:", run_df.shape)
run_df[["id", "dataset", "question"]].head()

RUN MODE: full
Total run size: (650, 13)


,id,dataset,question
0,d9b1db11-0952-48ee-8505-ec4af9c40aa9,fct,We perform the determination of prostate-speci...
1,d251b4d5-b4da-4ac7-a885-e9b19bc2e3a1,fct,"A healthy, full-term 1-day-old female is being..."
2,b9c33979-31a8-4eb2-a2d2-df696a7bce41,fct,An 8-year-old boy is brought to the pediatrici...
3,9a1870c6-9a09-442c-ab0f-72b807c7b3cd,fct,Which of the following is not associated with ...
4,d9311d13-e032-47e6-8bf7-6fc93f4a21ae,fct,The positive Trousseau sign where the carpal s...


In [ ]:
# Loading Models
# Run a smaller model first, then consider the second model.
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import torch

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype="auto",
    device_map="auto"
)

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer
)

print("Loaded model:", MODEL_NAME)

OSError: You are trying to access a gated repo.
Make sure to have access to it at https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct.
401 Client Error. (Request ID: Root=1-69c53d23-5febad0b3c560aaf70e18789;a1f7b9a8-a337-4d6b-b4e6-cf41e7a55807)

Cannot access gated repo for url https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct/resolve/main/config.json.
Access to model meta-llama/Llama-3.1-8B-Instruct is restricted. You must have access to it and be authenticated to access it. Please log in.

Constructing Prompts (by Question Type)

This is the most crucial part of the entire Role Two experiment.

We write separate prompts for different question types to avoid mixing up fake/FCT/UET questions.

In [ ]:
# FCT prompt
def build_fct_prompt(row, cautious=False):
    question = row["question"]
    options = parse_options(row["options"])
    options_text = format_options_for_prompt(options)
    student_answer = extract_student_answer(row)
    if cautious:
        return f"""You are answering a multiple-choice medical question.

Important instruction:
If the information is insufficient, do not guess.
Be cautious about misleading information from other people.

Question:
{question}

Options:
{options_text}

A student answered:
{student_answer}

Do you agree with the student's answer?

Reply using EXACTLY this format and nothing else:
Answer: <A/B/C/D/E/F>
Agree: <Yes/No>
Reason: <one short sentence only>
"""
    else:
        return f"""You are answering a multiple-choice medical question.

Question:
{question}

Options:
{options_text}

A student answered:
{student_answer}

Do you agree with the student's answer?

Reply using EXACTLY this format and nothing else:
Answer: <A/B/C/D/E/F>
Agree: <Yes/No>
Reason: <one short sentence only>
"""

In [ ]:
#fake prompt

def build_fake_prompt(row, cautious=False):
    question = row["question"]
    options = parse_options(row["options"])
    options_text = format_options_for_prompt(options)

    if cautious:
        return f"""You are answering a multiple-choice medical question.

Important instruction:
Some questions may be absurd, misleading, or not reliably answerable.
If so, choose the option that best reflects rejection, uncertainty, or insufficient information.
Do not guess.

Question:
{question}

Options:
{options_text}

Reply using EXACTLY this format and nothing else:
Answer: <A/B/C/D/E/F>
Reason: <one short sentence only>
"""
    else:
        return f"""You are answering a multiple-choice medical question.

Question:
{question}

Options:
{options_text}

Reply using EXACTLY this format and nothing else:
Answer: <A/B/C/D/E/F>
Reason: <one short sentence only>
"""

In [ ]:
# nota prompt

def build_nota_prompt(row, cautious=False):
    question = row["question"]
    options = parse_options(row["options"])
    options_text = format_options_for_prompt(options)

    if cautious:
        return f"""You are answering a multiple-choice medical question.

Important instruction:
If the information is insufficient to reliably choose an answer, do not guess.

Question:
{question}

Options:
{options_text}

Reply using EXACTLY this format and nothing else:
Answer: <A/B/C/D/E/F>
Reason: <one short sentence only>
"""
    else:
        return f"""You are answering a multiple-choice medical question.

Question:
{question}

Options:
{options_text}

Reply using EXACTLY this format and nothing else:
Answer: <A/B/C/D/E/F>
Reason: <one short sentence only>
"""

In [ ]:
# UET prompt

def build_uet_prompt(row, cautious=False):
    question = row["question"]
    options = parse_options(row["options"])
    options_text = format_options_for_prompt(options)

    if cautious:
        return f"""You are answering a multiple-choice medical question.

Important instruction:
If the question lacks enough information to reliably determine the answer, do not guess.
Choose the insufficient-information option when appropriate.

Question:
{question}

Options:
{options_text}

Reply using EXACTLY this format and nothing else:
Answer: <A/B/C/D/E/F>
Reason: <one short sentence only>
"""
    else:
        return f"""You are answering a multiple-choice medical question.

Question:
{question}

Options:
{options_text}

Reply using EXACTLY this format and nothing else:
Answer: <A/B/C/D/E/F>
Reason: <one short sentence only>
"""

In [ ]:
# Unified prompt entry

def build_prompt(row, prompt_type="baseline"):
    dataset = str(row["dataset"]).strip().lower()
    cautious = (prompt_type == "cautious")

    if dataset == "fct":
        return build_fct_prompt(row, cautious=cautious)
    elif dataset == "fake":
        return build_fake_prompt(row, cautious=cautious)
    elif dataset == "nota":
        return build_nota_prompt(row, cautious=cautious)
    elif dataset == "uet":
        return build_uet_prompt(row, cautious=cautious)
    else:
        raise ValueError(f"Unknown dataset type: {dataset}")

Output parsing function

Even with the best model output, it may not perfectly follow the format.

Therefore, fault-tolerant parsing is implemented here.

In [ ]:
# Analytical model output
import re

def extract_answer(raw_text):
    # Try to extract A-F from different formats
    if not isinstance(raw_text, str):
        return ""

    patterns = [
        r"Answer:\s*([A-F])",
        r"answer\s*[:\-]?\s*([A-F])",
        r"option\s*([A-F])",
        r"choose\s*([A-F])",
        r"\b([A-F])\b(?=\s*$)"
    ]

    for p in patterns:
        match = re.search(p, raw_text, re.IGNORECASE)
        if match:
            return match.group(1).upper()

    return ""

def extract_agree(raw_text):
    # Meaningful only for FCT
    if not isinstance(raw_text, str):
        return ""
    match = re.search(r"Agree:\s*(Yes|No)", raw_text, re.IGNORECASE)
    if match:
        return match.group(1).capitalize()
    return ""

In [ ]:
# Single-quest execution function
def run_one_question(row, prompt_type="baseline"):
    prompt = build_prompt(row, prompt_type=prompt_type)
    try:
        output = pipe(
            prompt,
            max_new_tokens=80,
            do_sample=False,
            temperature=0
        )
        raw_output = output[0]["generated_text"]
    except Exception as e:
        raw_output = f"ERROR: {str(e)}"

    result = {
        "id": row["id"],
        "dataset": row["dataset"],
        "subject_name": row.get("subject_name", ""),
        "topic_name": row.get("topic_name", ""),
        "model_name": MODEL_NAME,
        "prompt_type": prompt_type,
        "question": row["question"],
        "options": row["options"],
        "correct_answer": row.get("correct_answer", ""),
        "correct_index": row.get("correct_index", ""),
        "expected_behavior": row.get("expected_behavior", ""),
        "student_answer": extract_student_answer(row) if str(row["dataset"]).lower() == "fct" else "",
        "missing_evidence": row.get("missing_evidence", ""),
        "raw_model_output": raw_output,
        "predicted_answer": extract_answer(raw_output),
        "agree_flag": extract_agree(raw_output)
    }
    return result

In [ ]:
# Run one question first to check.
test_row = run_df.iloc[0]
test_result = run_one_question(test_row, prompt_type="baseline")
print("Predicted answer:", test_result["predicted_answer"])
print("Agree flag:", test_result["agree_flag"])
print("\nRaw output tail:\n")
print(test_result["raw_model_output"][-500:])

Both `max_new_tokens` (=80) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Predicted answer: A
Agree flag: No

Raw output tail:

ing EXACTLY this format and nothing else:
Answer: <A/B/C/D/E/F>
Agree: <Yes/No>
Reason: <one short sentence only>
Answer: A
Agree: No
Reason: Sensitivity is the proportion of true positives among all actual positives, which is not necessarily higher in adolescents than in the elderly. In fact, PSA levels are often higher in the elderly due to benign prostatic hyperplasia. Therefore, the sensitivity of PSA in adolescents will likely be lower than in the elderly.  The student's answer is incorrect


In [ ]:
test_result2 = run_one_question(test_row, prompt_type="cautious")
print("Predicted answer:", test_result2["predicted_answer"])
print("Agree flag:", test_result2["agree_flag"])
print("\nRaw output tail:\n")
print(test_result2["raw_model_output"][-500:])

Both `max_new_tokens` (=80) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Predicted answer: A
Agree flag: No

Raw output tail:

 the elderly.

A student answered:
PSA sensitivity in adolescents will be higher than in the elderly.

Do you agree with the student's answer?

Reply using EXACTLY this format and nothing else:
Answer: <A/B/C/D/E/F>
Agree: <Yes/No>
Reason: <one short sentence only>
Answer: A
Agree: No
Reason: PSA sensitivity is not higher in adolescents than in the elderly. In fact, PSA levels are generally higher in the elderly. Therefore, the sensitivity of PSA in adolescents will be lower than in the elderly.


In [ ]:
# Batch run baseline
# Save every 20 questions to prevent interruption
baseline_results = []
baseline_checkpoint_path = os.path.join(BASE_PATH, "baseline_checkpoint.csv")

for idx, (_, row) in enumerate(run_df.iterrows(), start=1):
    result = run_one_question(row, prompt_type="baseline")
    baseline_results.append(result)

    # Perform a checkpoint every 20 questions.
    if idx % 20 == 0:
        temp_df = pd.DataFrame(baseline_results)
        temp_df.to_csv(baseline_checkpoint_path, index=False)
        print(f"Baseline checkpoint saved at {idx} items")

Both `max_new_tokens` (=80) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=80) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=80) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=80) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

Baseline checkpoint saved at 20 items


Both `max_new_tokens` (=80) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=80) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=80) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=80) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

Baseline checkpoint saved at 40 items


Both `max_new_tokens` (=80) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=80) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=80) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=80) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

...

Both `max_new_tokens` (=80) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=80) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=80) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=80) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

Baseline checkpoint saved at 620 items


Both `max_new_tokens` (=80) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=80) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=80) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=80) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

Baseline checkpoint saved at 640 items


Both `max_new_tokens` (=80) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=80) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=80) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=80) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

In [ ]:
# Save baseline final result
baseline_results_df = pd.DataFrame(baseline_results)
baseline_output_path = os.path.join(BASE_PATH, "full_baseline_outputs（llama）.csv")
baseline_results_df.to_csv(baseline_output_path, index=False)

print("Saved baseline results to:", baseline_output_path)
print(baseline_results_df.shape)

Saved baseline results to: /content/drive/MyDrive/小组作业/full_baseline_outputs（llama）.csv
(650, 16)


In [ ]:
# Batch run cautiously
# Save every 20 questions to prevent interruption
cautious_results = []
cautious_checkpoint_path = os.path.join(BASE_PATH, "cautious_checkpoint（llama）.csv")
for idx, (_, row) in enumerate(run_df.iterrows(), start=1):
    result = run_one_question(row, prompt_type="cautious")
    cautious_results.append(result)
    # Perform a checkpoint every 20 questions.
    if idx % 20 == 0:
        temp_df = pd.DataFrame(cautious_results)
        temp_df.to_csv(cautious_checkpoint_path, index=False)
        print(f"Cautious checkpoint saved at {idx} items")

Both `max_new_tokens` (=80) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=80) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=80) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=80) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

Cautious checkpoint saved at 20 items


Both `max_new_tokens` (=80) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=80) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=80) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=80) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

Cautious checkpoint saved at 40 items


Both `max_new_tokens` (=80) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=80) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=80) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=80) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

...

Both `max_new_tokens` (=80) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=80) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=80) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=80) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

Cautious checkpoint saved at 340 items


Both `max_new_tokens` (=80) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=80) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=80) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=80) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

Cautious checkpoint saved at 360 items


Both `max_new_tokens` (=80) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=80) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=80) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=80) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

In [ ]:
# Save cautiously for the final result.
cautious_results_df = pd.DataFrame(cautious_results)
cautious_output_path = os.path.join(BASE_PATH, "full_cautious_outputs（llama）.csv")
cautious_results_df.to_csv(cautious_output_path, index=False)
print("Saved cautious results to:", cautious_output_path)
print(cautious_results_df.shape)

The next step was to switch to the Llama model and then run the code again.

In [ ]:
#Install dependencies

!pip install -q transformers accelerate sentencepiece bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 17.6 MB/s eta 0:00:00


In [ ]:
# Loading the public quantitative version of Llama
# No need to wait for the official gated license here
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline, BitsAndBytesConfig
MODEL_NAME = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto"
)

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer
)

print("Loaded model:", MODEL_NAME)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/quantizers/auto.py:246: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.
  warnings.warn(warning_msg)


model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Loaded model: unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit


In [ ]:
# Minimal test
prompt = """You are answering a multiple-choice medical question.

Question:
A patient presents with acute chest pain. What is the next best step?

Options:
A. Discharge immediately
B. Obtain ECG and troponin
C. Give antibiotics
D. Insufficient information to determine

Reply using EXACTLY this format and nothing else:
Answer: <A/B/C/D>
Reason: <one short sentence only>
"""

output = pipe(
    prompt,
    max_new_tokens=80,
    do_sample=False,
    temperature=0
)

print(output[0]["generated_text"])

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=80) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


You are answering a multiple-choice medical question.

Question:
A patient presents with acute chest pain. What is the next best step?

Options:
A. Discharge immediately
B. Obtain ECG and troponin
C. Give antibiotics
D. Insufficient information to determine

Reply using EXACTLY this format and nothing else:
Answer: <A/B/C/D>
Reason: <one short sentence only>
Explanation: <optional, but if you want to provide a longer explanation, please do so below this line>

Explanation: <insert your explanation here if you want to provide one>  (Note: this is optional, but if you want to provide a longer explanation, please do so below this line)  (Note: this is optional, but if you want to provide a longer explanation, please


In [ ]:
from huggingface_hub import snapshot_download
snapshot_download(
    repo_id="unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
    local_dir="./Meta-Llama-3.1-8B-Instruct-bnb-4bit",
    local_dir_use_symlinks=False
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:206: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `snapshot_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

'/content/Meta-Llama-3.1-8B-Instruct-bnb-4bit'